# **Predicting Electric Vehicle Purchases**

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# preprocess
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# model 
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Evaluation
from sklearn.metrics import accuracy_score, roc_curve, classification_report, confusion_matrix

# Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV


from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

In [24]:
train_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")
train_data.shape, test_data.shape

((668665, 15), (286571, 14))

### Data Prepare and Understand

In [25]:
train_data.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [26]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  object 
 9   City_Type                    668665 non-null  object 
 10  Current_Car_Type             668665 non-null  object 
 11  Home_Charging_Possible       668665 non-null  object 
 12  Subsidy_Available            668665 non-null  object 
 13 

In [27]:
# Check NULL
train_data.isnull().sum()

id                             0
Age                            0
Annual_Income_USD              0
Daily_Commute_km               0
Number_of_Cars_Owned           0
Charging_Stations_Near_Home    0
Charging_Stations_Near_Work    0
Environmental_Concern_Level    0
Gender                         0
City_Type                      0
Current_Car_Type               0
Home_Charging_Possible         0
Subsidy_Available              0
Range_Anxiety_Level            0
Will_Buy_EV                    0
dtype: int64

In [28]:
# Check DUPLICATES
train_data.duplicated().sum()

np.int64(0)

### Encoding

In [29]:
train_data.head(2)

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No


In [30]:
train_data.columns

Index(['id', 'Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender',
       'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level', 'Will_Buy_EV'],
      dtype='object')

In [31]:
encode_col = ['Gender',
       'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level']

for col in encode_col:
    print(f"{col}:-  {train_data[col].unique()}")


Gender:-  ['Male' 'Female' 'Other']
City_Type:-  ['Suburban' 'Rural' 'Urban']
Current_Car_Type:-  ['Sedan' 'SUV' 'Hatchback' 'Truck']
Home_Charging_Possible:-  ['Yes' 'No']
Subsidy_Available:-  ['No' 'Yes']
Range_Anxiety_Level:-  ['Low' 'Medium' 'High']


In [32]:
def Encoder_Func(data):

    mappings = {
        "City_Type": {
            "Suburban": 0,
            "Rural": 1,
            "Urban": 2
        },

        "Current_Car_Type": {
            "Sedan": 0,
            "SUV": 1,
            "Hatchback": 2,
            "Truck": 3
        },

        "Range_Anxiety_Level": {
            "Low": 0,
            "Medium": 1,
            "High": 2
        },

        "Gender": {
            "Male": 0,
            "Female": 1,
            "Other": 2
        },

        "Home_Charging_Possible": {
            "Yes": 1,
            "No": 0
        },

        "Subsidy_Available": {
            "Yes": 1,
            "No": 0
        }

    }

    for col, mapping in mappings.items():
        data[col] = data[col].map(mapping)

    return data

train_data["Will_Buy_EV"] = train_data['Will_Buy_EV'].map({"No": 0, "Yes": 1})
train_data = Encoder_Func(train_data)
train_data.head(2)

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,0,0,0,1,0,0,0
1,1,38,30000.0,5.0,1,2,2,4.0,0,1,1,1,0,0,0


### **Features Extraction**

In [33]:
def add_features(X):
    X = X.copy()
 
    # Income is right-skewed and acts multiplicatively -> log scale
    X["log_income"] = np.log1p(X["Annual_Income_USD"])
 
    # Overall charging ecosystem the person can rely on
    X["total_charging_stations"] = (
        X["Charging_Stations_Near_Home"] + X["Charging_Stations_Near_Work"]
    )
 
    # Range anxiety matters more the longer the commute
    X["commute_x_range_anxiety"] = X["Daily_Commute_km"] * X["Range_Anxiety_Level"]
 
    # A subsidy changes the decision more for buyers with less money
    # (with log income this is a smooth "how much does subsidy matter at this income" term)
    X["subsidy_x_log_income"] = X["Subsidy_Available"] * X["log_income"]
 
    # Willingness (concern) x ability (income) to pay
    X["env_x_log_income"] = X["Environmental_Concern_Level"] * X["log_income"]
 
    # raw income is replaced by log_income (avoids a redundant near-duplicate)
    return X.drop(columns=["Annual_Income_USD"])
train_data = add_features(train_data)

train_data.head()

,id,Age,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,log_income,total_charging_stations,commute_x_range_anxiety,subsidy_x_log_income,env_x_log_income
0,0,66,23.4,2,3,7,1.0,0,0,0,1,0,0,0,11.439150,10,0.0,0.00000,11.439150
1,1,38,5.0,1,2,2,4.0,0,1,1,1,0,0,0,10.308986,4,0.0,0.00000,41.235944
2,2,26,36.8,1,8,15,5.0,1,2,0,0,1,0,1,11.455190,23,0.0,11.45519,57.275952
3,3,66,23.7,2,6,9,3.0,0,0,2,1,0,0,0,11.206142,15,0.0,0.00000,33.618426
4,4,54,50.8,1,2,3,3.0,0,0,2,1,0,0,0,10.966455,5,0.0,0.00000,32.899366


### Data Scaling

In [35]:
x = train_data.drop(columns=["id", "Will_Buy_EV"], axis=1)
y = train_data['Will_Buy_EV'].values

In [36]:
ss = StandardScaler()
ss.fit(x)
x = ss.transform(x)

joblib.dump(ss, "scaler.joblib")
print("Scaler SAVE")

Scaler SAVE


### **Data Emblanced handling**

In [37]:
train_data['Will_Buy_EV'].value_counts()

Will_Buy_EV
0    551886
1    116779
Name: count, dtype: int64

In [38]:
# # UnderSampling
# und_sam = RandomUnderSampler(random_state=42)
# und_x, und_y = und_sam.fit_resample(x, y)

In [39]:
# OverSampling
ovr_sam = RandomOverSampler(random_state=42)
ovr_x, ovr_y = ovr_sam.fit_resample(x, y)

### Data Spliting

In [40]:
x_train, x_test, y_train, y_test = train_test_split(
        ovr_x, ovr_y,
        test_size=0.1,
        random_state=42
)

print("Train:", x_train.shape, y_train.shape)
print("Test:", x_test.shape,  y_test.shape)

Train: (993394, 17) (993394,)
Test: (110378, 17) (110378,)


## Build Model

In [41]:
rf = RandomForestClassifier()
rf.fit(x_train, y_train)

RandomForestClassifier()

In [42]:
y_prd = rf.predict(x_test)
print(accuracy_score(y_test, y_prd))
(confusion_matrix(y_test, y_prd) / len(y_test))*100


0.9574643497798474


array([[45.68845241,  4.15934335],
       [ 0.09422167, 50.05798257]])

In [43]:
xgb = XGBClassifier()
xgb.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [44]:
y_prd = xgb.predict(x_test)
print(accuracy_score(y_test, y_prd))
print((confusion_matrix(y_test, y_prd) / len(y_test))*100)

0.8814437659678559
[[42.01018319  7.83761257]
 [ 4.01801084 46.13419341]]


In [ ]:
sfs

NameError: name 'sfs' is not defined

# **Submission**

In [ ]:
id = test_data['id']

x_data = Encoder_Func(test_data)
x_data = ss.transform(x_data)

#final_prd = rf.predict_proba(x_data)[:, 1]

In [ ]:
p = prd.ravel()
p.shape

In [ ]:
submission = pd.DataFrame(data={"id": id, "Will_Buy_EV": prd.ravel()})
submission.to_csv("submission_neural_network.csv", index=False)